# 4-3절 연습 문제 풀이

이 노트북은 4-3절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch04/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# MNIST 데이터셋 준비 (내려받기 경로는 저장소의 download 디렉터리)
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

DATA_ROOT = '../../download'

def mnist_loaders(batch_size=64, transform=None, valid_ratio=0.2):
    transform = transform or transforms.ToTensor()
    full = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
    n_valid = int(len(full) * valid_ratio)
    g = torch.Generator().manual_seed(SEED)
    train_set, valid_set = random_split(full, [len(full) - n_valid, n_valid], generator=g)
    return (DataLoader(train_set, batch_size=batch_size, shuffle=True),
            DataLoader(valid_set, batch_size=batch_size),
            DataLoader(test_set, batch_size=batch_size))

def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss = correct = n = 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            loss = criterion(out, labels)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (out.argmax(dim=1) == labels).sum().item()
            n += labels.size(0)
    return total_loss / n, correct / n * 100

## 연습 4-8

CIFAR-10 데이터셋은 MNIST만큼이나 딥러닝 교육에 많이 사용되는 유명한 데이터셋으로, torchvision.datasets의 CIFAR10 클래스를 통해 파이토치 내장 데이터셋으로 제공된다. MNIST 데이터셋처럼 10개의 클래스로 구성되어 있는데, 이미지 샘플이 32x32 크기의 3채널 컬러 이미지(사진)라는 점이 다르다.

CIFAR-10 데이터셋에서 하나의 이미지 샘플은 몇 개의 숫자로 표현될까?

CIFAR-10 데이터셋으로 다층 퍼셉트론 모델을 학습한다고 가정하고 데이터셋과 데이터로더를 만들어 보자(표준화 기준 평균과 표준편차는 모든 채널에 0.5를 사용한다).

In [ ]:
from torchvision import datasets, transforms
cifar = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True,
                         transform=transforms.ToTensor())
image, label = cifar[0]
print(f'이미지 텐서 형태: {tuple(image.shape)}  (채널, 세로, 가로)')
print(f'숫자 개수: 3 x 32 x 32 = {image.numel()}개')
print(f'MNIST(1 x 28 x 28)의 {image.numel() / 784:.2f}배')

CIFAR-10 이미지 한 장은 **3 × 32 × 32 = 3,072개**의 숫자로 표현된다. MNIST(784개)의 약 3.9배다. 채널이 3개로 늘고 해상도도 커졌기 때문인데, 이 차이가 다층 퍼셉트론으로는 한계가 있는 이유이자 5장 합성곱 신경망이 필요한 이유다.

## 연습 4-9

[코드 4-21]의 데이터 변환 객체에는 이미지 크기 변환을 설명하기 위해 굳이 28x28 이미지를 20x20으로 줄이는 transforms.Resize가 포함되어 있었고, 숫자 분류기 모델을 만들 때는 이를 제외했다. 이번에는 제외하지 않은 데이터 변환 객체를 사용해 데이터셋을 만들고, 이 데이터셋을 사용해 숫자 분류기 모델도 만들어 보자.

힌트: 입력 이미지의 크기가 바뀌면 모델도 그에 맞춰 바뀌어야 한다.

In [ ]:
# 20x20으로 줄이는 Resize를 포함한 변환 객체
transform_resize = transforms.Compose([
    transforms.Resize((20, 20)),
    transforms.ToTensor(),
])
train_loader, valid_loader, test_loader = mnist_loaders(transform=transform_resize)

images, _ = next(iter(train_loader))
print(f'입력 이미지 형태: {tuple(images.shape)}')

# 입력 크기가 바뀌었으므로 첫 선형 계층의 입력 크기도 400(20x20)으로 바꾼다.
model = nn.Sequential(nn.Flatten(), nn.Linear(20 * 20, 128), nn.ReLU(),
                      nn.Linear(128, 10)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1, 6):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, valid_loader, criterion)
    print(f'{epoch}/5 훈련 {tr_loss:.4f} / 검증 {va_loss:.4f} ({va_acc:.2f}%)')
print(f'평가 정확도: {run_epoch(model, test_loader, criterion)[1]:.2f}%')

입력 이미지가 28×28에서 20×20으로 줄면 평탄화 결과가 784에서 **400**으로 바뀌므로, 첫 선형 계층의 입력 크기도 함께 바꿔야 한다. 정보량이 줄어 정확도는 조금 떨어지지만 파라미터 수와 학습 시간은 줄어든다.

## 연습 4-10

256개와 128개의 뉴런을 가진 두 개의 은닉층을 사용한 이번 절의 다층 퍼셉트론 모델의 구조를 바꿔 가며 실험해, 검증 손실과 평가 정확도가 더 좋은 모델을 만들어 보자. 실험 과정에서 중요한 재현 가능성도 염두에 두고 진행하자.

In [ ]:
def build_mlp(hidden_sizes, dropout=0.0):
    layers = [nn.Flatten()]
    fan_in = 784
    for h in hidden_sizes:
        layers += [nn.Linear(fan_in, h), nn.ReLU()]
        if dropout: layers.append(nn.Dropout(dropout))
        fan_in = h
    layers.append(nn.Linear(fan_in, 10))
    return nn.Sequential(*layers)

train_loader, valid_loader, test_loader = mnist_loaders()
configs = [([256, 128], 0.0), ([512, 256], 0.0), ([256, 128], 0.2), ([128], 0.0)]
for hidden, dr in configs:
    torch.manual_seed(SEED)        # 재현 가능성 - 매 실험마다 같은 시드로 초기화
    model = build_mlp(hidden, dr).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    best = float('inf'); best_acc = 0
    for epoch in range(5):
        run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, valid_loader, criterion)
        if vl < best: best, best_acc = vl, va
    te = run_epoch(model, test_loader, criterion)[1]
    print(f'은닉층 {str(hidden):12s} 드롭아웃 {dr:.1f} -> '
          f'검증 손실 {best:.4f}, 평가 정확도 {te:.2f}%')

실험에서 가장 중요한 것은 **재현 가능성**이다. 매 실험 직전 같은 시드로 초기화해야 구조 차이만 비교할 수 있다. 시드를 고정하지 않으면 초깃값 차이 때문에 어떤 구조가 나은지 판단할 수 없다.

일반적으로 은닉층을 키우면 훈련 손실은 계속 줄지만 검증 손실은 어느 지점부터 나빠진다(과적합). 드롭아웃은 그 격차를 줄여 준다.

## 연습 4-11

숫자 분류기 모델의 분류 결과로부터 10x10 형태의 혼동 행렬 텐서를 만드는 함수를 작성하고, 혼동 행렬을 출력해 보자.

참고로 많은 데이터 분석 라이브러리에 혼동 행렬 계산 기능이 포함되어 있다. 하지만, 여기서는 텐서와 관련된 파이토치 함수와 메서드만 사용해 도전해 보길 권한다.

In [ ]:
# 파이토치 텐서 연산만으로 10x10 혼동 행렬을 만든다.
def confusion_matrix(preds, targets, n_classes=10):
    # 정답 * 클래스 수 + 예측 -> 0 ~ 99 사이의 고유 번호로 만든 뒤 개수를 센다.
    pair = targets * n_classes + preds
    counts = torch.bincount(pair, minlength=n_classes ** 2)
    return counts.reshape(n_classes, n_classes)     # 행: 정답, 열: 예측

train_loader, valid_loader, test_loader = mnist_loaders()
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                      nn.Linear(128, 10)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(5):
    run_epoch(model, train_loader, criterion, optimizer)

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for images, labels in test_loader:
        all_preds.append(model(images.to(device)).argmax(dim=1).cpu())
        all_targets.append(labels)
cm = confusion_matrix(torch.cat(all_preds), torch.cat(all_targets))

print('     ' + ''.join(f'{i:5d}' for i in range(10)) + '   <- 예측')
for i, row in enumerate(cm):
    print(f'{i:3d} |' + ''.join(f'{v:5d}' for v in row.tolist()))
print('\n^ 정답')
print(f'\n정확도: {cm.diag().sum().item() / cm.sum().item() * 100:.2f}%')

`정답 × 클래스 수 + 예측`으로 (정답, 예측) 쌍을 0~99의 고유 번호로 바꾼 뒤 `bincount()`로 세고, 10×10으로 재구성하는 것이 핵심이다. 대각선은 맞힌 개수, 대각선 밖은 혼동된 개수다.

> 공통 라이브러리의 `viz.plot_confusion_matrix(cm)`로 그림으로도 확인할 수 있다.

## 연습 4-12

[도전 문제] 순서대로 계이름을 나열하면 음악도 분석 가능한 데이터가 된다. 다음은 동요 <반짝반짝 작은별>의 계이름(쉼표 포함)으로, 도레미파솔라시를 각각 0123456으로, 쉼표를 7로 표현한 것이다.

004455473322110744332217443322170044554733221107

이 데이터를 학습해서 8개의 계이름(쉼표 포함)을 입력하면 다음 음 또는 쉼표를 예측하는 다층 퍼셉트론 모델을 만들어 보자. 순서가 있는 데이터(순차 데이터)를 처리하는 모델은 6장에서 소개하지만, 결과가 좋지 않더라도 상관없으니 일단 4장까지 학습한 내용을 총동원해 도전해 보자.

4장 학습 노트

In [ ]:
# <반짝반짝 작은별> 계이름 데이터 - 8개를 보고 다음 음을 예측
SEQ = '004455473322110744332217443322170044554733221107'
notes = torch.tensor([int(c) for c in SEQ])
WINDOW, N_NOTE = 8, 8          # 0~6 계이름 + 7 쉼표

xs = torch.stack([notes[i:i + WINDOW] for i in range(len(notes) - WINDOW)])
ys = notes[WINDOW:]
X = nn.functional.one_hot(xs, N_NOTE).float().flatten(1)    # (N, 8*8)
print(f'샘플 {len(X)}개, 입력 크기 {X.shape[1]}')

torch.manual_seed(SEED)
model = nn.Sequential(nn.Linear(WINDOW * N_NOTE, 64), nn.ReLU(),
                      nn.Linear(64, N_NOTE))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1000):
    loss = criterion(model(X), ys)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

acc = ((model(X).argmax(dim=1) == ys).float().mean() * 100).item()
print(f'학습 정확도: {acc:.2f}%')

# 마중물 8음으로 이어서 16음 생성
seed_notes = [int(c) for c in SEQ[:WINDOW]]
generated = list(seed_notes)
for _ in range(16):
    window = torch.tensor(generated[-WINDOW:])
    x = nn.functional.one_hot(window, N_NOTE).float().flatten().unsqueeze(0)
    generated.append(model(x).argmax(dim=1).item())
names = '도레미파솔라시_'
print(f'입력:  {"".join(names[n] for n in seed_notes)}')
print(f'생성:  {"".join(names[n] for n in generated[WINDOW:])}')

계이름을 원-핫 벡터로 바꿔 8음을 이어 붙이면 (8×8=64) 크기의 입력이 된다. 다음 음을 맞히는 **다중 클래스 분류** 문제로 바꿔 푸는 것이 요령이다.

다만 이 방식은 정해진 길이(8음)만 볼 수 있고 순서 정보를 직접 다루지 못한다. 이 한계를 구조적으로 푸는 방법이 6장의 순환 신경망이며, 실제로 6-2절에서 같은 <작은별> 데이터를 다시 사용한다.